## Q1. Generating questions

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
filenames = (
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
)
sample_docs = [doc for doc in documents if doc["filename"] in filenames]

In [4]:
sample_docs

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [5]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI

zai_client = OpenAI(
    api_key=os.getenv("ZAI_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/"
)

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:

import json

from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        zai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="GLM-4.7-Flash"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [10]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(sample_docs[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
avg_usage = sum(u.prompt_tokens for u in usages) / len(usages)
avg_usage

1446.6666666666667

## Q2. First result with text search

In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
chunk_texts = [doc["filename"] + " " + doc["content"] for doc in chunks]

In [18]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(chunks)

In [22]:
from minsearch import VectorSearch

from tqdm.auto import tqdm
import numpy as np

from embedder import Embedder

embed = Embedder()

batch_size = 50
X = []

for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

  0%|          | 0/6 [00:00<?, ?it/s]

In [28]:
def text_search(query, num_results=5):
    boost_dict = {"content": 2.0}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [31]:
def vector_search(query, num_results=5):
    vquery = embed.encode(query)

    return vindex.search(
        vquery,
        num_results=num_results
    )

In [24]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [25]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [26]:
q = ground_truth[0]["question"]

In [34]:
text_results = text_search(q, num_results=5)
text_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Q3. First result with vector search

In [33]:
vector_results = vector_search(q, num_results=5)
vector_results

[{'start': 1000,
  'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they somet

## Q4. Evaluating text search

In [35]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [36]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [45]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [39]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [43]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [46]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5. Evaluating vector search

In [47]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7277777777777777, 'mrr': 0.5469907407407407}

## Q6. Tuning hybrid search

In [49]:
for k in [1, 50, 100, 200]:
    print(f"Evaluating hybrid search with k={k}...")
    results = evaluate(ground_truth, lambda query: hybrid_search(query, k=k))
    print(f"Results for k={k}: {results}")

Evaluating hybrid search with k=1...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=1: {'hit_rate': 0.8194444444444444, 'mrr': 0.6486111111111111}
Evaluating hybrid search with k=50...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
Evaluating hybrid search with k=100...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
Evaluating hybrid search with k=200...


  0%|          | 0/360 [00:00<?, ?it/s]

Results for k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
